# Capítulo 4 — Avaliando o sistema e escolhendo o modelo

## Missão
Transformar as métricas do capítulo 3 num **pipeline de avaliação** do assistente: um conjunto de perguntas com referência, uma diretriz com critérios, verificadores automáticos de instruções, custo e latência por modelo, intervalo de confiança e análise por fatias. No fim, você escolhe o modelo com dados, não com opinião.

**No fim deste caderno você terá:** o módulo `meu_assistente/avaliacao.py`, que avalia qualquer função `responder` e devolve uma tabela com nota, F1, similaridade, latência e custo.

**Leitura essencial (opcional, livro em inglês):**
- Evaluation Criteria (Instruction-Following, Cost and Latency) · PDF p. 184
- Model Selection Workflow · PDF p. 203
- Design Your Evaluation Pipeline · PDF p. 224

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap04_avaliando_o_sistema_e_escolhendo_o_modelo.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
Avaliar um modelo solto não basta: o que importa é o **sistema** (prompt + contexto + modelo + regras) no **seu** caso de uso. O caminho é:
1. escrever o que é uma boa resposta (diretriz);
2. montar um conjunto de avaliação parecido com o uso real;
3. medir com vários métodos;
4. olhar os resultados por fatias e com margem de erro.

Rankings públicos ajudam só a fazer a primeira peneira.

### 4.1 **Verificadores de instrução.** Escreva 4 funções que conferem uma resposta e devolvem `True/False`: `max_palavras(t, n)`, `contem(t, termo)`, `eh_json(t)` e `termina_com_pergunta(t)`. Peça ao modelo: "Em no máximo 25 palavras, explique o plano Grátis da Nimbus, cite 'CSV' e termine com uma pergunta ao cliente." Passe a resposta pelos verificadores que valem para esse pedido.

### 4.2 **Custo e latência por modelo.** Para 3 perguntas de `PERGUNTAS`, rode `responder` com `MODELO` e com `MODELO_FORTE`. Monte uma tabela com a média de latência e de tokens por modelo e um custo estimado (use os preços do capítulo 1).

### 4.3 **Diretriz de avaliação.** Escreva a diretriz do assistente como um dicionário de critérios, cada um com `descricao` e `peso` (pesos somam 1): por exemplo correção, uso só da documentação, tom e concisão. Escreva `nota_final(notas)`, que recebe notas de 1 a 5 por critério e devolve a média ponderada.

### 4.4 **Pipeline de avaliação.** Escreva `avaliacao.py` com `avaliar(responder, perguntas, juiz=True)`. Para cada item de `perguntas`, chama `responder(pergunta)` (que devolve um dict com `texto`, `latencia_s`, `tokens_in`, `tokens_out` e `modelo`) e calcula: `f1`, `semantica`, `nota_juiz` (se `juiz=True`), latência e custo (dicionário `PRECOS` por modelo). Devolve um `DataFrame` que mantém também o `doc` de cada pergunta. Inclua `resumo(df)`, com as médias, a % de notas >= 4 e a latência p90.

_A célula grava `meu_assistente/avaliacao.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/avaliacao.py


### 4.5 **Linha de base.** Avalie o `nucleo.responder` (sem documentação, só o modelo) nas 8 primeiras perguntas. Mostre a tabela e o `resumo`. Guarde o resultado em `../saidas/base_cap04.csv`: você vai comparar com o RAG no capítulo 6.

### 4.6 **Margem de erro (bootstrap).** Com poucas perguntas, a média engana. Faça 2.000 reamostragens com reposição das notas do juiz e mostre o intervalo de 95% (percentis 2,5 e 97,5) da média.

### 4.7 **Fatias.** Agrupe o resultado por `doc` (assunto da pergunta) e veja em qual assunto o sistema vai pior. É ali que vale investir primeiro.

### 4.8 **Seleção de modelo.** Avalie as mesmas 8 perguntas com `MODELO_FORTE` (use `lambda p: responder(p, modelo=MODELO_FORTE)`). Compare os dois resumos lado a lado e calcule **nota média por dólar**. Qual você escolheria para o suporte, e por quê?

### 4.9 **Aumente o conjunto de avaliação.** Poucas perguntas dão pouca confiança. Para 3 itens de `PERGUNTAS`, peça ao modelo 2 reformulações de cada um (JSON `{"perguntas": [...]}`), mantendo a referência e o `doc`. Junte numa lista `PERGUNTAS_AMPLIADAS` e salve em `../saidas/perguntas_ampliadas.json`. Depois, confira as reformulações à mão.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
